### 복습 
- 가전 폴더 안에 모든 데이터파일을 로드해서 하나의 데이터프레임으로 생성 
- 감정에 대한 데이터들이 3개 분류 -> 2개의 분류로 변경 (부정, 중립 -> 부정)
- 감정 데이터가 없는 데이터들은 따로 저장 
- train, test의 비율은 8:2
- Dataset을 기존의 Dataset 구성과 같이 작업
- RawText 데이터를 이용하여 감정분석 모델을 생성 
- SBERT 모델을 이용하여 임베딩 
- 다중퍼셉트론의 모델을 이용하여 감정 분석 (Linear -> ReLU -> DropOut -> Linear)
- 검증 데이터를 이용하여 정확도와 f1_score 확인 
- 감정 데이터가 없는 RawText에서 sample(10)를 출력하여 감정 예측

- 다중퍼셉트론 모델이 아닌 머신러닝 모델 (SVC)을 이용하여 감정 분석 예측

In [2]:
import os 
import re
import pandas as pd 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 문자 정규화 함수 정의 
def normailize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [4]:
# 파일의 목록을 로드 -> 목록을 기준으로 데이터를 로드 -> 단순 결합 
file_path = "../data/가전/"
file_list = os.listdir(file_path)
file_list

['3-1.영상음향가전(76).json',
 '3-1.영상음향가전(77).json',
 '3-1.영상음향가전(78).json',
 '3-1.영상음향가전(79).json',
 '3-1.영상음향가전(80).json',
 '3-1.영상음향가전(81).json',
 '3-1.영상음향가전(82).json',
 '3-1.영상음향가전(83).json',
 '3-1.영상음향가전(84).json',
 '3-1.영상음향가전(85).json',
 '3-1.영상음향가전(86).json',
 '3-1.영상음향가전(87).json',
 '3-1.영상음향가전(88).json',
 '3-2.생활미용욕실가전(128).json',
 '3-2.생활미용욕실가전(129).json',
 '3-2.생활미용욕실가전(130).json',
 '3-2.생활미용욕실가전(131).json',
 '3-2.생활미용욕실가전(132).json',
 '3-2.생활미용욕실가전(133).json',
 '3-2.생활미용욕실가전(134).json',
 '3-2.생활미용욕실가전(135).json',
 '3-2.생활미용욕실가전(136).json',
 '3-2.생활미용욕실가전(137).json',
 '3-2.생활미용욕실가전(138).json',
 '3-2.생활미용욕실가전(139).json',
 '3-2.생활미용욕실가전(140).json',
 '3-3.주방가전(127).json',
 '3-3.주방가전(128).json',
 '3-3.주방가전(129).json',
 '3-3.주방가전(130).json',
 '3-3.주방가전(131).json',
 '3-3.주방가전(132).json',
 '3-3.주방가전(133).json',
 '3-3.주방가전(134).json',
 '3-3.주방가전(135).json',
 '3-3.주방가전(136).json',
 '3-3.주방가전(137).json',
 '3-3.주방가전(138).json',
 '3-3.주방가전(139).json',
 '3-4.계절가전(126).json',
 '3-4.계절가전(127)

In [5]:
# 로드한 데이터프레임을 누적으로 결합하기 위해 빈 데이터프레임 생성 
df = pd.DataFrame()

for file in file_list:
    # file : 파일명 
    data = pd.read_json(file_path + file)
    # df에 단순 행 결합 -> df에 다시 대입 
    df = pd.concat( [df, data], axis= 0  )
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 411.9+ KB


In [6]:
# 필요한 컬럼을 제외하고 나머지 컬럼을 무시 
df = df[['RawText', 'GeneralPolarity']]

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RawText          4056 non-null   object 
 1   GeneralPolarity  3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 95.1+ KB


In [8]:
df.rename(columns = {
    'GeneralPolarity' : 'label'
}, inplace=True)

In [9]:
# RawText의 문자 정규화 사용
df['RawText'] = df['RawText'].map(normailize)

In [10]:
# RawText에서 길이가 1 이하인 데이터를 제외
df = df.loc[df['RawText'].str.len() > 1]

In [11]:
# 혹시 RawText에 중복 데이터가 존재할수 있으니 중복 제거 
df.drop_duplicates(subset='RawText', inplace=True)

In [12]:
# label이 결측치인 데이터는 따로 저장 
na_df = df.loc[df['label'].isna(), ]
na_df

,RawText,label
13,귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만 재질만 바꾸...,NaN
43,아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거워서 떨어지...,NaN
44,화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나 하지 않았...,NaN
55,이번에 이사하면서 우리 따님께서 방에 TV가 있으면 좋겠다고 하여 방에서 사용할 T...,NaN
93,기존에 사용하던 무선이어폰이 오래되어서 배터리가 광탈하는 바람에 새로운 상품이 필요...,NaN
...,...,...
41,대용량이고 세척이 간편하다는 얘기에 구매를 했는데 저는 별로인 거 같아요...생각했...,NaN
44,요거 진짜 진짜 물건입니다. 처음엔 디자인이 너무 귀여워서 주문하게 되었는데요. 작...,NaN
68,지인의 추천으로 믿고 바로 구매를 해서 현재도 사용중입니다좀 더 많은 분들에게 도움...,NaN
76,다른 에어쿨러와 다르게 슬림한 디자인이 마음에 들어요.심플한 디자인 덕분에 집안 어...,NaN


In [ ]:
# df에는 결측치를 제외
df = df.loc[ ~df['label'].isna(), ]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 86.2+ KB


In [14]:
df.reset_index(drop=True, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3678 entries, 0 to 3677
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 57.6+ KB


In [15]:
# label의 데이터의 빈도수를 확인 
df['label'].value_counts()

label
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [16]:
# label의 데이터들을 int형태로 변경 
df['label'] = df['label'].astype(int)

In [17]:
df['label'].value_counts()

label
 1    2220
 0     944
-1     514
Name: count, dtype: int64

In [18]:
# 삼중 분류의 class를 이진 분류로 변경하기 위해 -1을 0으로 변경 
df['label'] = df['label'].map(
    {
        -1 : 0, 
        0 : 1, 
        1 : 2
    }
)

In [19]:
df['label'].value_counts()

label
2    2220
1     944
0     514
Name: count, dtype: int64

In [20]:
# train, test 로 분할
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

In [21]:
train_df['label'].value_counts()

label
2    1776
1     755
0     411
Name: count, dtype: int64

In [22]:
model_name = 'BM-K/KoSimCSE-roberta-multitask'
sbert = SentenceTransformer(model_name)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [23]:
# 최대 토큰 길이 제한 
sbert.max_seq_length = 128

In [24]:
class SBERTHead(Dataset):
    def __init__(self, texts, labels):
        with torch.inference_mode():
            self.emb = sbert.encode(
                texts, convert_to_tensor=True, normalize_embeddings=True
            )
            self.labels = torch.tensor(labels, dtype=torch.long)
        self.texts = texts
        self.labels2 = labels
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]
        # getitem에서 임베딩 처리 
        # res_emb = sbert.encode(
        #     self.texts[idx], convert_to_tensor=True, normalize_embeddings=True
        # )
        # res_label = torch.tensor(self.labels2[idx], dtype=torch.long)
        # return res_emb, res_label

In [45]:
train_ds = SBERTHead(train_df['RawText'].tolist(), train_df['label'].tolist())
test_ds = SBERTHead(test_df['RawText'].tolist(), test_df['label'].tolist())

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=256)

In [83]:
# 다중 퍼셉트론 구조의 분류 모델 생성 
class MLPHead(nn.Module):
    def __init__(self, input_dim, hidden = 256, num_classes = 2, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(), 
            nn.Dropout(dropout), 
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x):
        result = self.net(x)
        return result

In [85]:
# sbert 모델에서 출력 차원의 개수를 확인
in_dim = sbert.get_sentence_embedding_dimension()
clf = MLPHead(in_dim, num_classes=3)
# 손실 함수 -> 데이터의 불균형 문제는 해결하기 위해 class 별 가중치 변화를 각각 부여
# [0, 1, 2] -> 적은 데이터일수록 높은 변화
# 5~10 : 데이터의 비율이 굉장히 적은 경우
# 2.0 ~ 4.0 : 데이터의 비율이 적은 경우
# 1.0 : 가장 많은 데이터비율 
crit = nn.CrossEntropyLoss(weight= torch.tensor([5.0, 3.0, 1.0]))
# AdamW -> Adam 개선판
opt = optim.AdamW(clf.parameters(), lr=2e-4)

In [86]:
clf.train()

for epoch in range(5):
    total = 0.0
    for x, y in train_dl:
        opt.zero_grad()
        logits = clf(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    print(f"epoch : {epoch}, loss : {round(total/len(train_df), 4)}")

epoch : 0, loss : 1.0828
epoch : 1, loss : 1.0434
epoch : 2, loss : 0.9923
epoch : 3, loss : 0.9331
epoch : 4, loss : 0.8711


In [87]:
# 정확도, f1-score 확인 
clf.eval()

y_true , y_pred = [], []

with torch.inference_mode():
    for x, y in test_dl:
        logits = clf(x)
        pred = logits.argmax(dim = 1).tolist()
        y_true.extend(y.tolist())
        y_pred += pred


In [92]:

print('accuracy_score', accuracy_score(y_true, y_pred))
print('f1_score', f1_score(y_true, y_pred, average='macro'))

accuracy_score 0.6182065217391305
f1_score 0.632558620765116


In [94]:
samples = na_df['RawText'].sample(15).tolist()
samples

['핫해서 줄 서서 사야하는 화이트와 그린을 사이에 두고 고민을 좀 하다 평소 애정 하는 그린으로 선택했습니다캠핑장에선 그린색이 자연 친화적이라는 .... 워터저그는 보온 보냉력이 상당히 좋다고 하는데 일반 물통보다 3배 두꺼운 단열폼으로 메뉴얼상 보온 13시간 보냉 60시간이 가능하다고 합니다. 잠금방식도 견고하고 출수구는 방향 전환이 가능하고 쉽게 분해가 되어 세척이 간편하더라구요 출수구 누름이 부드럽고 유속도 좋은데 조절도 가능해서 캠핑 때 물로 인한 스트레스는 이제 없을 거 같아요 시리즈는 저를 실망시키지 않아서 너무 좋아요',
 '이 제품은 저를 위한 겁니다 머리를 말리는 거야 대충 물기야 사라져라 하는데... 항상 앞머리가 문제더라고요.앞머리가 제대로 안 나오면 그날은 그냥 아침부터 망쳐진 느낌이에요. 그만큼 앞머리가 중요하더라고요. 그런데 이 제품 사용하면서 앞머리 관리하는 게 훨씬 편해졌어요.전 판 고데기는 별로 쓰지 않고 락 스위치 걸고 봉 고데기로만 사용하고 있어요. 이렇게 락 스위치가 있어서 봉으로 사용 가능한 고데기는 처음이네요.아침 등교 때 앞머리용으로 구매한 고데기인데 흡족해요. 주말에 시간 여유 있을 땐 락 스위치 풀고 판 고데기로 머리에 컬도 한번 시도할까 봐요.',
 '영화감상과 음악감상까지 쉽고 편리하게 사용할 수 있는 제품을 고민하다가 검색해서 이 제품 패키지를 골랐는데 결과적으로 기대이상의 대만족입니다. 가성비를 생각해서 av 와 음감겸용 시스템을 비교해 보면 이만한 제품도 없다고 생각합니다. 특히 일반적인 사운드바 형태의 제품들과 다른 점은 웬만한 OOO급의 타워 스피커로 디자인도 뛰어나지만 전체적인 음질에 중요한 역할을 담당하고 있다는 겁니다. 특히 타워 스피커로 음감시 만족도가 높습니다. 홈씨어터로 최고인 거 같습니다.',
 '주방에 무거운 대형 믹서기만 있어서 슬림한 블랜더를 찾다가 홈쇼핑에 나오는 요 제품에 꽂혀서 결제했네요..일명 우선 색상이 레드 민트 블루 등 화사한 색상이 많아서 좋았고 그 중에 저는 은은한 파

In [95]:
@torch.no_grad()
def predict_review(texts, batch_size = 128):
    if isinstance(texts, str):
        texts = [texts]
    texts_norm = [normailize(t) for t in texts]  

    sbert.eval()
    clf.eval()

    result = []

    for idx in range(0, len(texts), batch_size):
        batch_text = texts_norm[idx : idx + batch_size]
        embs = sbert.encode(
            batch_text, convert_to_tensor = True, 
            normalize_embeddings = True
        )

        logits = clf(embs)
        probs = logits.softmax(dim = -1)
        preds = probs.argmax(dim = -1).tolist()
        id2label = {
            0 : '부정', 
            1 : '중립',
            2 : '긍정'
        }
        for idx2, pred in enumerate(preds):
            prob = float(probs[idx2, pred])
            review = texts[idx + idx2]
            label = id2label[pred]
            result.append(
                {
                    'text' : review, 
                    'prob' : prob, 
                    'label' : label
                }
            )
    return result

In [ ]:
predict_review(samples)[:10]